In [1]:
import cv2 
import os
import numpy as np
import tensorflow as tf 
from PIL import Image
from sklearn.model_selection import train_test_split
from keras.utils import normalize, to_categorical
from keras.models import Sequential
from keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D
from keras.applications import ResNet50
from keras.applications.resnet50 import preprocess_input
from keras.optimizers import Adam
from google.colab import drive, files
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [2]:
# 1. Kết nối Drive (Chỉ cần làm 1 lần mỗi buổi)
drive.mount('/content/drive')

# 2. Giải nén dữ liệu (Dùng os.system để tránh lỗi "Expected expression" trong VS Code)
zip_path = "/content/drive/MyDrive/Colab_test/dataset.zip"
if not os.path.exists('/content/dataset'):
    print("Đang giải nén dữ liệu trên server Google...")
    os.system(f'unzip -q "{zip_path}" -d "/content/"')
    print("Giải nén hoàn tất!")

#Cấu hình các tham số
INPUT_SIZE = 224 
image_directory = "/content/dataset/" # Sử dụng đường dẫn trên máy ảo Linux
dataset = []
label = []

no_tumor_images = os.listdir(image_directory + 'no/')
yes_tumor_images = os.listdir(image_directory + 'yes/')

for i, image_name in enumerate(no_tumor_images):
    if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        full_path = os.path.join(image_directory, 'no', image_name)
        image = cv2.imread(full_path)
        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)   
            image = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE))     
            dataset.append(image)
            label.append(0) 

for i, image_name in enumerate(yes_tumor_images):
    if image_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        full_path = os.path.join(image_directory, 'yes', image_name)
        image = cv2.imread(full_path)
        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (INPUT_SIZE, INPUT_SIZE))
            dataset.append(image)
            label.append(1)

dataset = np.array(dataset)
label = np.array(label)

print(f"\nĐã nạp xong: {len(dataset)} ảnh. Kích thước mảng: {dataset.shape}")

Mounted at /content/drive
Đang giải nén dữ liệu trên server Google...
Giải nén hoàn tất!

Đã nạp xong: 3000 ảnh. Kích thước mảng: (3000, 224, 224, 3)


In [3]:
#chia -> 3 tập + chuẩn hoá
x_temp, x_test, y_temp, y_test = train_test_split(dataset, label, test_size=0.15, random_state=42, shuffle=True)
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.176, random_state=42, shuffle=True)
# gemini gen chuẩn hoá sử dụng preprocess
x_train = preprocess_input(x_train.astype('float32')) 
x_val = preprocess_input(x_val.astype('float32'))
x_test = preprocess_input(x_test.astype('float32'))

In [4]:
import gc

# Chỉ xóa nếu biến đã tồn tại trong RAM
if 'dataset' in locals():
    del dataset
if 'label' in locals():
    del label

# Thu gom rác để giải phóng RAM cho Tesla T4
gc.collect()
print("✅ Đã làm sạch bộ nhớ thành công!")

✅ Đã làm sạch bộ nhớ thành công!


In [5]:
#one-hot encode
y_train = to_categorical(y_train, num_classes = 2)
y_val = to_categorical(y_val, num_classes = 2)
y_test = to_categorical(y_test, num_classes = 2)

In [ ]:
#data_augmentation (off)
datagen = ImageDataGenerator(
    rotation_range=15,      
    width_shift_range=0.1,  
    height_shift_range=0.1, 
    shear_range=0.1,        
    zoom_range=0.1,         
    horizontal_flip=True,   
    fill_mode='nearest'
)

In [7]:
#khởi tạo mô hình + tuỳ chỉnh tham số
model = ResNet50(weights='imagenet', include_top=False, input_shape=(INPUT_SIZE, INPUT_SIZE, 3))

# Đóng băng các lớp của mô hình base
for layer in model.layers[:-15]: # để dư 10 layers cuối để mô hình tự cải thiện 
    layer.trainable = False

# Tạo mô hình mới với các lớp tùy chỉnh
model = Sequential([
    model,
    #GlobalAveragePooling2D(), # Sử dụng pooling thay vì flatten 
    Flatten(),
    Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [8]:
#khởi chạy
model.compile(
    optimizer = Adam(learning_rate=0.0001), 
    loss = 'categorical_crossentropy', 
    metrics = ['accuracy']
    )
model.fit(
    datagen.flow(x_train, y_train, batch_size=32),
    epochs = 30, 
    verbose = 1,
    validation_data = (x_val, y_val),
    shuffle = True # loại bỏ học theo thứ tự, trộn 2 tập yes + no random để model up khả năng ứng biến
    )

Epoch 1/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 64s 670ms/step - accuracy: 0.8505 - loss: 8.7113 - val_accuracy: 0.9555 - val_loss: 6.3359
Epoch 2/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 28s 426ms/step - accuracy: 0.9648 - loss: 5.6177 - val_accuracy: 0.9777 - val_loss: 4.9853
Epoch 3/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 28s 428ms/step - accuracy: 0.9724 - loss: 4.5572 - val_accuracy: 0.9822 - val_loss: 4.1293
Epoch 4/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 28s 427ms/step - accuracy: 0.9805 - loss: 3.7927 - val_accuracy: 0.9822 - val_loss: 3.4771
Epoch 5/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 28s 428ms/step - accuracy: 0.9852 - loss: 3.1977 - val_accuracy: 0.9866 - val_loss: 2.9460
Epoch 6/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 29s 432ms/step - accuracy: 0.9848 - loss: 2.7350 - val_accuracy: 0.9733 - val_loss: 2.5624
Epoch 7/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 41s 425ms/step - accuracy: 0.9829 - loss: 2.3837 - val_accuracy: 0.9800 - val_loss: 2.2058
Epoch 8/30
66/66 ━━━━━━━━━━━━━━━━━━━━ 29s 435ms/step - accuracy: 0.9862 - loss: 2.0583 - val_accu

In [9]:
print("\n--- BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP TEST ---")
loss, accuracy = model.evaluate(x_test, y_test)
print(f"🎯 Độ chính xác THỰC TẾ (Unbiased Accuracy): {accuracy * 100:.2f}%")


--- BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP TEST ---
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 295ms/step - accuracy: 0.9978 - loss: 0.2941
🎯 Độ chính xác THỰC TẾ (Unbiased Accuracy): 99.78%


In [10]:
# 1. Yêu cầu Google kết nối lại Drive (phòng trường hợp nó bị ngắt ngầm)
drive.mount('/content/drive')

# 2. Đường dẫn chuẩn xác để lưu thẳng vào ổ Google Drive của bạn
duong_dan_drive = '/content/drive/MyDrive/Final.keras'

# 3. Lưu mô hình!
model.save(duong_dan_drive)
print(f"✅ ĐÃ ĐÓNG GÓI VÀ GỬI LÊN GOOGLE DRIVE THÀNH CÔNG!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ ĐÃ ĐÓNG GÓI VÀ GỬI LÊN GOOGLE DRIVE THÀNH CÔNG!
